# MARV on Qwen2.5-0.5B (Colab T4)

Everything here stays at **0.5B**. Three checkpoints, one architecture (24 layers,
hidden 896, intermediate 4864), so `marv.diff` works across every pair:

| checkpoint | role |
|---|---|
| `Qwen/Qwen2.5-0.5B-Instruct` | the live model we browse, edit, and measure |
| `Qwen/Qwen2.5-0.5B` | pretrained base — diff target for *what post-training moved* |
| `Qwen/Qwen2.5-Coder-0.5B` | code continued-pretraining from the same base — diff target for *what code training moved* |

Flow: extract a **vindex** -> browse -> locate a fact's constellation *causally* ->
edit it and measure the collateral -> sweep the Pareto frontier -> two weight-space diffs.

Runtime: **T4 GPU** (`Runtime > Change runtime type > T4`). First cell downloads ~1 GB.

In [ ]:
!pip install -q 'transformers>=4.45' accelerate safetensors matplotlib
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

## Load + extract

`extract` copies the FFN gate/down + embed/unembed + final-norm into numpy.
`build_down_meta(device=...)` precomputes every feature's promoted tokens once
(`lm_head @ down`) on the GPU — seconds even at Qwen's 152k vocab; the numpy path
is minutes. We save the vindex straight away so a later session can skip re-extracting.

In [ ]:
import torch, numpy as np, gc, marv
from transformers import AutoModelForCausalLM, AutoTokenizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'

INSTRUCT = 'Qwen/Qwen2.5-0.5B-Instruct'
BASE     = 'Qwen/Qwen2.5-0.5B'
CODER    = 'Qwen/Qwen2.5-Coder-0.5B'

tok = AutoTokenizer.from_pretrained(INSTRUCT)
model = AutoModelForCausalLM.from_pretrained(INSTRUCT, torch_dtype=torch.float16).to(device).eval()
print(model.config.num_hidden_layers, 'layers  hidden', model.config.hidden_size,
      ' intermediate', model.config.intermediate_size, ' vocab', model.config.vocab_size)

vindex = marv.extract(model, model_name=INSTRUCT)
marv.build_down_meta(vindex, device=device)
vindex.save('/content/qwen25-0.5b-instruct.vindex.npz')
print('bands:', vindex.layer_bands)

## Browse: `describe_entity`

Bare-embedding gate-KNN + logit-lens across the knowledge band. No forward pass.
You get `feature -> promoted tokens`, not a knowledge graph.

In [ ]:
for entity in ['France', 'Germany', 'Japan', 'Einstein']:
    print(f'\n=== {entity} ===')
    for r in marv.describe_entity(vindex, tok, entity, k_features=4):
        print('  ', r)

## Contextual constellation

The bare embedding is a weak query on most models. Pass the live `model` + the real
fact prompt, differenced against a baseline, to query the model's *actual* hidden
state — much sharper. This is the candidate pool for the causal step below.

In [ ]:
pool = marv.constellation(vindex, tok, 'France', model=model,
                          prompt='The capital of France is',
                          baseline_prompt='The capital of',
                          per_layer=8, device=device)
for r in pool[:12]:
    print(f'  L{r.layer:>2} f{r.feature:<5} sim={r.sim:.2f}  -> {r.tokens[:3]}')

## The battery

`target` carries rephrasings (an edit that only works on the exact wording has poor
generalization). `neighbour` shares France's constellation — collateral risk.
`control` spans geo / science / lexical / math / history so damage anywhere shows up.

> This is a **demo-sized** battery (~30 probes). A real edit eval wants 100+ controls
> across >=5 sub-domains — collateral rate is a proportion and its error bar is
> `sqrt(p(1-p)/n)`, bigger than the signal at n=10.

In [ ]:
P = marv.Probe
battery = [
    P('The capital of France is', 'Paris', ('target',)),
    P('The French capital is', 'Paris', ('target',)),
    P('Paris is the capital of', 'France', ('target',)),
    P('What is the capital of France? It is', 'Paris', ('target',)),

    P('The capital of Italy is', 'Rome', ('neighbour','capital')),
    P('The capital of Spain is', 'Madrid', ('neighbour','capital')),
    P('The capital of Germany is', 'Berlin', ('neighbour','capital')),
    P('The capital of Portugal is', 'Lisbon', ('neighbour','capital')),
    P('The official language of France is', 'French', ('neighbour','france')),
    P('The currency used in France is the', 'euro', ('neighbour','france')),
    P('The Eiffel Tower is in', 'Paris', ('neighbour','paris')),
    P('The Louvre is in', 'Paris', ('neighbour','paris')),

    P('The capital of Japan is', 'Tokyo', ('control','geo')),
    P('The capital of Egypt is', 'Cairo', ('control','geo')),
    P('The capital of Canada is', 'Ottawa', ('control','geo')),
    P('The capital of Brazil is', 'Brasilia', ('control','geo')),
    P('The largest planet in the solar system is', 'Jupiter', ('control','science')),
    P('Water is made of hydrogen and', 'oxygen', ('control','science')),
    P('The chemical symbol for gold is', 'Au', ('control','science')),
    P('The speed of light is approximately 300,000', 'kilometers', ('control','science')),
    P('The opposite of hot is', 'cold', ('control','lexical')),
    P('The past tense of go is', 'went', ('control','lexical')),
    P('The plural of mouse is', 'mice', ('control','lexical')),
    P('Two plus two equals', 'four', ('control','math')),
    P('Ten minus three equals', 'seven', ('control','math')),
    P('The author of Romeo and Juliet is', 'Shakespeare', ('control','history')),
    P('The first president of the United States was', 'George', ('control','history')),
    P('World War II ended in the year', '1945', ('control','history')),
]
print(len(battery), 'probes')

## Baseline — keep only facts the model already knows

An edit eval is meaningless on a fact the model gets wrong unedited. Drop anything
the model doesn't rank in its top 3.

In [ ]:
base = marv.run_battery(model, tok, battery, device=device)
for r in base.rows:
    m = 'ok' if r.target_rank == 1 else f'r{r.target_rank}'
    print(f'  [{m:>4}] p={r.target_prob:.2f}  {r.prompt!r} -> {r.top1!r}  {r.tags}')

known = {r.prompt for r in base.rows if r.target_rank <= 3}
battery = [p for p in battery if p.prompt in known]
print(f'\nkept {len(battery)} / {len(base.rows)} probes (target rank <= 3)')

## Locate the constellation — causally

1. the contextual gate-KNN above gave a **candidate pool**.
2. `rank_by_ablation_effect` suppresses each candidate *alone* and ranks by the
   measured drop in target probability — the causal constellation, not a geometric proxy.

In [ ]:
target_probes = [p for p in battery if 'target' in p.tags]
ranked = marv.rank_by_ablation_effect(
    model, tok, [(r.layer, r.feature) for r in pool[:30]], target_probes, device=device)
for (L, f), drop in ranked[:15]:
    tks, _ = marv.describe_feature(vindex, L, f, k=3)
    words = [w.strip() for w in tok.batch_decode([[int(t)] for t in tks])]
    print(f'  L{L:>2} f{f:<5} drop={drop:+.3f}  -> {words}')
feats = [c for c, _ in ranked]

## One edit: suppress the top 5

`suppress` zeros those MLP activations on the **live model** via forward hooks
(reversible). `study_edit` runs the battery before/after and reports what moved,
grouped by tag.

In [ ]:
rep = marv.study_edit(model, tok, marv.suppress(model, feats[:5]), battery, device=device)
rep.show()
print()
for tag, m in sorted(rep.metrics().items()):
    print(f'  {tag:<14} n={int(m["n"]):>2}  moved={m["moved"]:.2f}  mean dprob={m["mean_dprob"]:+.3f}')

## Which layers of the constellation carry the fact?

The constellation spans several layers, but they don't carry `France -> Paris`
equally. `suppression_by_layer` suppresses **one layer's slice at a time** and
diffs the battery: the layers whose slice drops the target are where the fact
lives; layers with features in the constellation but ~zero target effect were
geometric KNN hits, not causal.

`cumulative=True` adds layers shallow -> deep so you can see the depth at which
the edit's effect saturates.

In [ ]:
from marv.evaluate import suppression_by_layer

edit_feats = feats[:8]   # a slightly wider constellation so it spans >1 layer

print('--- each layer in isolation ---')
print(f'{"L":>3} {"n":>2} {"features":>16}  {"target dp":>10} {"neigh dp":>10} {"ctrl dp":>10}')
iso = suppression_by_layer(model, tok, edit_feats, battery, device=device)
for L, fs, d in iso:
    m = d.metrics()
    g = lambda t: m.get(t, {}).get('mean_dprob', 0.0)
    print(f'{L:>3} {len(fs):>2} {str(list(fs)):>16}  '
          f'{g("target"):>+10.3f} {g("neighbour"):>+10.3f} {g("control"):>+10.3f}')

print('\n--- cumulative, shallow -> deep ---')
print(f'{"<=L":>3}  {"target dp":>10} {"neigh dp":>10} {"ctrl dp":>10}')
for L, fs, d in suppression_by_layer(model, tok, edit_feats, battery, device=device, cumulative=True):
    m = d.metrics()
    g = lambda t: m.get(t, {}).get('mean_dprob', 0.0)
    print(f'{L:>3}  {g("target"):>+10.3f} {g("neighbour"):>+10.3f} {g("control"):>+10.3f}')

In [ ]:
# bar chart: per-layer efficacy vs collateral of the constellation edit
import matplotlib.pyplot as plt

Ls = [L for L, _, _ in iso]
eff  = [-iso_d.metrics().get('target',    {}).get('mean_dprob', 0.0) for _, _, iso_d in iso]
coll = [-iso_d.metrics().get('neighbour', {}).get('mean_dprob', 0.0) for _, _, iso_d in iso]

x = range(len(Ls))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - 0.2 for i in x], eff,  width=0.4, label='target drop (efficacy)')
ax.bar([i + 0.2 for i in x], coll, width=0.4, label='neighbour drop (collateral)')
ax.set_xticks(list(x)); ax.set_xticklabels([f'L{L}' for L in Ls])
ax.set_ylabel('mean prob drop when only this layer is suppressed')
ax.set_title('per-layer contribution of the France->Paris constellation')
ax.legend(); ax.grid(alpha=.3, axis='y')
plt.tight_layout(); plt.show()

## The Pareto frontier

Sweep constellation size. For each `n`, suppress `feats[:n]` and diff the battery.
Plot target-prob drop (efficacy) against neighbour/control drop (collateral).

In [ ]:
sizes = [0, 1, 2, 3, 4, 6, 8, 10, 14, 20]
sweep = marv.suppression_frontier(model, tok, feats, battery, sizes=sizes, device=device)
rows = []
for n, d in sweep:
    m = d.metrics()
    g = lambda t, k: m.get(t, {}).get(k, 0.0)
    rows.append((n, g('target','mean_dprob'), g('neighbour','mean_dprob'), g('control','mean_dprob'),
                 g('neighbour','moved'), g('control','moved')))
print(f'{"n":>3} {"target dp":>10} {"neigh dp":>10} {"ctrl dp":>10} {"neigh mv":>9} {"ctrl mv":>8}')
for r in rows:
    print(f'{r[0]:>3} {r[1]:>+10.3f} {r[2]:>+10.3f} {r[3]:>+10.3f} {r[4]:>9.2f} {r[5]:>8.2f}')

In [ ]:
import matplotlib.pyplot as plt
ns = [r[0] for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ns, [-r[1] for r in rows], 'o-', label='target (efficacy)')
ax[0].plot(ns, [-r[2] for r in rows], 's-', label='neighbour (collateral)')
ax[0].plot(ns, [-r[3] for r in rows], '^-', label='control (collateral)')
ax[0].set_xlabel('features suppressed'); ax[0].set_ylabel('mean target-prob drop')
ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot([-r[2] for r in rows], [-r[1] for r in rows], 'o-')
for r in rows:
    ax[1].annotate(str(r[0]), (-r[2], -r[1]), fontsize=8, xytext=(3, 3), textcoords='offset points')
ax[1].set_xlabel('neighbour prob drop (collateral)'); ax[1].set_ylabel('target prob drop (efficacy)')
ax[1].set_title('Pareto frontier'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## suppress vs ablate vs steer

- `suppress` — forward hooks, reversible.
- `ablate` — zeros `down_proj[:, f]` in the weights (every path); should track `suppress`.
- `steer` — adds `alpha * -embed(' Paris')` to the residual mid-knowledge-band; usually blunter.

In [ ]:
ef = feats[:5]
sup = marv.study_edit(model, tok, marv.suppress(model, ef), battery, device=device).metrics()

b0 = marv.run_battery(model, tok, battery, device=device)
saved = marv.ablate(model, ef)
abl = marv.diff_battery(b0, marv.run_battery(model, tok, battery, device=device)).metrics()
marv.restore(model, saved)

pid = tok.encode(' Paris', add_special_tokens=False)[0]
kb = vindex.band('knowledge'); mid = kb[len(kb) // 2]
ste = marv.study_edit(model, tok,
                      marv.steer(model, mid, vindex.embed[pid].astype(np.float32), alpha=-10.0),
                      battery, device=device).metrics()

print(f'{"":<10} {"target dp":>10} {"neigh dp":>10} {"ctrl dp":>10}')
for name, m in [('suppress', sup), ('ablate', abl), ('steer', ste)]:
    g = lambda t: m.get(t, {}).get('mean_dprob', 0.0)
    print(f'{name:<10} {g("target"):>+10.3f} {g("neighbour"):>+10.3f} {g("control"):>+10.3f}')

## Weight-space diff 1 — base vs instruct

`Qwen2.5-0.5B` (base) vs `Qwen2.5-0.5B-Instruct`, feature by feature: which neurons
post-training moved, and whether it changed what they *fire on* (`gate_cos`) or what
they *promote* (`down_cos`). Free the live model first to keep T4 host RAM happy.

In [ ]:
del model; gc.collect(); torch.cuda.empty_cache()

base_m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16)
vindex_base = marv.extract(base_m, model_name=BASE)
marv.build_down_meta(vindex_base, device=device)
vindex_base.save('/content/qwen25-0.5b-base.vindex.npz')
del base_m; gc.collect()

deltas_it = marv.diff(vindex_base, vindex)
print('most-moved features, post-training:')
for d in marv.most_changed(deltas_it, k=12):
    b, _ = marv.describe_feature(vindex_base, d.layer, d.feature_idx, k=3)
    a, _ = marv.describe_feature(vindex,      d.layer, d.feature_idx, k=3)
    bw = [w.strip() for w in tok.batch_decode([[int(t)] for t in b])]
    aw = [w.strip() for w in tok.batch_decode([[int(t)] for t in a])]
    print(f'  L{d.layer:>2} f{d.feature_idx:<5} gate_cos={d.gate_cos_sim:+.3f} '
          f'down_cos={d.down_cos_sim:+.3f}  {bw} -> {aw}')

In [ ]:
scores = marv.per_layer_score(deltas_it, metric='mean_topk')
print('layers that absorbed the most change (post-training):')
for L in sorted(scores, key=lambda l: -scores[l])[:8]:
    print(f'  L{L:>2}: {scores[L]:.4f}')

## Weight-space diff 2 — base vs Coder

`Qwen2.5-Coder-0.5B` is code continued-pretraining from the *same* `Qwen2.5-0.5B`
base, so `diff(base, coder)` is the clean "what did code training move" picture —
and you can compare its layer profile against post-training's above.

_(drop the instruct vindex first — reload it from the `.npz` if you want it back)_

In [ ]:
del vindex; gc.collect()

coder_m = AutoModelForCausalLM.from_pretrained(CODER, torch_dtype=torch.float16)
vindex_coder = marv.extract(coder_m, model_name=CODER)
marv.build_down_meta(vindex_coder, device=device)
vindex_coder.save('/content/qwen25-0.5b-coder.vindex.npz')
del coder_m; gc.collect()

deltas_code = marv.diff(vindex_base, vindex_coder)
print('most-moved features, code training:')
for d in marv.most_changed(deltas_code, k=12):
    b, _ = marv.describe_feature(vindex_base,  d.layer, d.feature_idx, k=3)
    a, _ = marv.describe_feature(vindex_coder, d.layer, d.feature_idx, k=3)
    bw = [w.strip() for w in tok.batch_decode([[int(t)] for t in b])]
    aw = [w.strip() for w in tok.batch_decode([[int(t)] for t in a])]
    print(f'  L{d.layer:>2} f{d.feature_idx:<5} gate_cos={d.gate_cos_sim:+.3f} '
          f'down_cos={d.down_cos_sim:+.3f}  {bw} -> {aw}')

In [ ]:
import numpy as np
s_it   = marv.per_layer_score(deltas_it,   metric='mean_topk')
s_code = marv.per_layer_score(deltas_code, metric='mean_topk')
layers = sorted(s_it)
a = np.array([s_it[l]   for l in layers])
b = np.array([s_code[l] for l in layers])
print(f'per-layer weight change correlation (post-training vs code): {np.corrcoef(a, b)[0,1]:.3f}')
print(f'{"L":>3} {"post-train":>11} {"code":>9}')
for l in layers:
    print(f'{l:>3} {s_it[l]:>11.4f} {s_code[l]:>9.4f}')

## Reading it

- **Frontier knee** (target drops while neighbours stay flat) = a clean edit exists;
  operate there. **Straight line through the origin** = the fact and its neighbours
  share features 1:1, no clean suppression edit — try `steer` or accept the trade.
- `ablate` should track `suppress`. `steer` is usually blunter on both axes.
- The two diffs answer different questions on the same base: post-training reshapes
  the **output/formatting** band more; code training tends to move **mid-band** features
  (tokenization + syntax). Compare the per-layer tables.

### Try next
- Swap `France` for `Liechtenstein` or a fictional place — collateral usually collapses
  because the constellation is barely shared.
- Widen the battery to 100+ controls before trusting the third decimal of any metric.
- `marv.diff(vindex, quantized_vindex)` — which features 4-bit quantization breaks.
- Reload any vindex with `marv.VindexLite.load('/content/qwen25-0.5b-*.vindex.npz')`
  to skip re-extracting next session.